In [ ]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain_qdrant import QdrantVectorStore


from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from qdrant_client import QdrantClient,models

import os
import openai
from tqdm import tqdm
import pickle
import re
from datasets import Dataset
from ragas import evaluate

In [ ]:
openai.api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
openai_emb = OpenAIEmbeddings(openai_api_key=os.environ["OPENAI_API_KEY"],model="text-embedding-ada-002")
openai_model =  ChatOpenAI(openai_api_key=os.environ["OPENAI_API_KEY"],model="gpt-4o-mini", temperature=0.15)

In [ ]:
TITLE = 'Skull King'

# Dataset Creation
Creo un dataset di query chiedendo un LLM di generare domande sul gioco scelto partendo dai chunk

In [ ]:
filter = models.Filter(
        must=[
            models.FieldCondition(
                key='metadata.title',
                match=models.MatchValue(value=TITLE),
            )
        ]
    )

In [ ]:
#creo una lista di chunk con relativi metadati
client = QdrantClient(url="192.168.1.7:6333")
points = client.scroll(collection_name="bg",scroll_filter=filter)

chunks = [] 
while points:
    for point in points[0]:
        payload = point.payload
        chunks.append({"page_content":payload["page_content"],"metadata":payload["metadata"]})
    if points[1]:
        points = client.scroll(collection_name="bg",offset=points[1],scroll_filter=filter)
    else:
        break

In [ ]:
len(chunks)

In [ ]:
#setting LLM
template = """
You are an assistant specialized in analyzing texts for board games.
You will be provided with an excerpt of text (chunk) related to the rules, descriptions, or other elements of a board game, along with its contextual metadata.
Your task is to generate a set of questions based exclusively on the rules of the game described in the chunk and metadata, ensuring the questions are strictly related to the game and not to any other unrelated topics that might appear in the text.

Instructions:
 - Generate a variable number of questions depending on the complexity and amount of information in the chunk.
 - he questions must concern only the rules, mechanics, or elements of the game described in the input. Ignore any other unrelated topics that may appear in the text.
 - If the chunk is too short or does not contain sufficient information about the game?s rules, return "No questions can be generated."
 - Do not add, invent, or infer anything that is not explicitly stated in the chunk and metadata.
 - Formulate clear, concise, and relevant questions that help clarify or explore the game?s rules and mechanics.
Provided Input:
 - Text Chunk: {page_content}
 - Metadata: {metadata}
Expected Output:
 - A list of relevant questions based solely on the game?s rules and mechanics as described in the provided chunk and metadata.
 - If no questions can be generated, return: "No questions can be generated."

"""

prompt = ChatPromptTemplate.from_template(template)



chain = (
    {"page_content":RunnablePassthrough(), "metadata": RunnablePassthrough()}
    | prompt
    | openai_model
    | StrOutputParser()
)


In [ ]:
file_name_queries = "_".join(el.lower() for el in TITLE.split(" "))+"_queries.pkl"
file_name_answer = "_".join(el.lower() for el in TITLE.split(" "))+"_answer.pkl"

In [ ]:
queries = []
for chunk in tqdm(chunks):
    res = chunk
    resp = chain.invoke({"page_content":chunk["page_content"],"metadata":str(chunk["metadata"])})
    res["queries"] = resp
    queries.append(res)

In [ ]:
# with open("skull_king_queries.pkl","wb") as f:
#     pickle.dump(queries,f)

# Answers generation

In [ ]:
with open("skull_king_queries.pkl","rb") as f:
    queries = pickle.load(f)

In [ ]:
queries

In [ ]:
qdrant = QdrantVectorStore.from_existing_collection(collection_name="bg",embedding=openai_emb,url="192.168.1.7:6333")
retriever = qdrant.as_retriever(search_kwargs={"k":5,"fetch_k":10,"lambda_mult":0.9},search_type="mmr")

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc for doc in docs)


template = """
You are an expert assistant specializing in board games. Your role is to provide authoritative, precise, and practical guidance on game rules, mechanics, strategies, and scenarios. 
You respond as the ultimate reference for the games discussed, ensuring clarity and correctness. Your answers should feel as though they?re guiding the player through a live game session. 
Avoid general advice or unrelated topics. Instead, focus entirely on providing rule explanations, strategic insights, and in-game examples based on the player's current scenario.

The game you're explaining today is: **{title}**

---
**Current Situation**:  
This is the specific context or scenario the player is in, which might affect your answer:  
_{context}_

---
**Player's Question**:  
_{question}_

---
**Response**:  
Provide your answer in an instructive and conversational tone as if you?re explaining the rules and strategies at the table. Include relevant examples, clarify mechanics, and offer advice on how to best handle the current scenario:

- **Game Rule Explanation**: Offer precise details on the relevant game rules, mechanics, or actions related to the question.
  
- **Contextual Strategy/Advice**: If applicable, give strategic advice based on the player?s current in-game context, During this contextualization, do not give example of a specific case, just be vague for some strategy applicable in general, not in the specific case, unless explicity asked so.

- **Example**: Where useful, provide an example to illustrate the explanation more clearly.
"""

prompt = ChatPromptTemplate.from_template(template)



chain = (
    {"title":RunnablePassthrough(), "context": RunnablePassthrough() | format_docs, "question": RunnablePassthrough()}
    | prompt
    | openai_model
    | StrOutputParser()
)

In [ ]:
answers = []
for query in tqdm(queries):
    questions_string = query["queries"]
    questions = questions_string.split("\n")
    for question in questions:
        if question == "No questions can be generated.":
            continue
        question = re.sub(r"([0-9]\. )","",question)
        contexts = [doc.page_content for doc in retriever.invoke(question)]
        answer = chain.invoke({"title":"Skull King", "context": contexts, "question": question})
        answers.append({"contexts": contexts, "question": question,"answer":answer})

In [ ]:
# with open("skull_king_answers.pkl","wb") as f:
#     pickle.dump(answers,f)

In [ ]:
eval_dataset = Dataset.from_list(answers)

In [ ]:
from ragas.metrics import LLMContextPrecisionWithoutReference,ContextEntityRecall,Faithfulness,AnswerRelevancy,AnswerSimilarity,AnswerCorrectness

In [ ]:
metrics = [LLMContextPrecisionWithoutReference(),Faithfulness(),AnswerRelevancy()]

In [ ]:
results = evaluate(dataset=eval_dataset,metrics=metrics,llm=openai_model,embeddings=openai_emb)

In [ ]:
df = results.to_pandas()

In [ ]:
df.to_csv("results.csv")

In [ ]:
df

In [ ]:
df.describe()

In [ ]:
from evaluation import generate_eval_dataset
from vector_db import QdrantVectorDB

In [ ]:
URL = ""
API_KEY = ""

In [ ]:
db = QdrantVectorDB(URL,API_KEY)

In [ ]:
generate_eval_dataset(model=openai_model,emb=openai_emb,db=db,collection_name="bg",title="Skull King")